# Підготовка середовища та структури (Task 0)

Перш ніж писати агента, ми підготуємо структуру нашого проєкту. Правильна архітектура — це запорука того, що наш код не перетвориться на хаос у Jupyter Notebook.

## 1. Створення структури
Ми розділяємо проєкт на дві основні частини:
- `core/` — "мозок" нашого агента (Pydantic-моделі та чиста логіка). Цей пакет нічого не знає про інтернет чи LLM.
- `connectors/` — "руки" та "очі" агента (доступ до DOU, виклик Gemini, надсилання листів).

Також ми створили теки:
- `helpers/` — для допоміжного коду (наприклад, складного HTTP-запиту на DOU з куками), щоб не перевантажувати вебінар.
- `data/` — для зберігання вашого PDF-резюме.
- `fixtures/` — для кешованих даних.

## 2. Залежності
Ми використовуємо `uv` (сучасний менеджер пакетів) для встановлення бібліотек:
`uv add pydantic pypdf httpx google-genai instructor beautifulsoup4 python-dotenv`

- `pydantic` — для створення структурованих моделей даних.
- `google-genai` + `instructor` — для роботи з LLM та отримання чіткого JSON з тексту.
- `httpx` + `beautifulsoup4` — для завантаження та парсингу вакансій з DOU.
- `pypdf` — для читання резюме.

## 3. Секрети
Створено файл `.env.example`. Зробіть його копію з назвою `.env` та заповніть своїми даними (ключ Gemini, пароль додатку Gmail, куки DOU). Файл `.env` ігнорується Git-ом.

In [ ]:
# Додаємо корінь проєкту в sys.path, щоб наші пакети core/ та connectors/ імпортувалися коректно
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent)) # Вказуємо на корінь проєкту (батьківська директорія для notebooks/)

from dotenv import load_dotenv
load_dotenv()  # підтягує GOOGLE_API_KEY та інші секрети з файлу .env

print("Оточення завантажено успішно!")

# Читання резюме: PDF → текст (Task 1)

Перший крок нашого агента — зрозуміти, хто ви такі. Зазвичай резюме зберігається у форматі PDF. 
На цьому етапі нам не потрібно зберігати гарне форматування, колонки чи шрифти. Наше завдання — просто витягти "сирий" текст. LLM чудово вміє читати навіть неструктурований, суцільний текст без абзаців.

Ми використаємо `pypdf` — легку бібліотеку на чистому Python без зайвих системних залежностей.

In [ ]:
from connectors.resume_loader import load_resume_text

# Завантажуємо тестове резюме з папки data
# (Перед цим переконайтеся, що ви поклали свій PDF-файл у data////)
resume_text = load_resume_text("../data/Kostiantyn_Zivenko_Resume_EN.pdf")

print("Довжина тексту:", len(resume_text))
print("\nПерші 500 символів:\n", resume_text[:500])

## Очищення тексту (Sanitizing)

Ми отримали сирий текст, але він може містити артефакти парсингу: зайві пробіли, кілька перенесень рядків підряд або дивні символи з PDF (наприклад, маркери списків перетворюються на `\uf0b7`).
Для LLM це не завжди критично, але краще трохи "почистити" текст. Це:
1. Зменшить розмір тексту (кількість токенів — це вартість і швидкість).
2. Зробить його більш читабельним.

In [ ]:
import re

def sanitize_text(text: str) -> str:
    # 1. Видаляємо спецсимволи маркерів з PDF (наприклад \uf0b7)
    text = text.replace('\uf0b7', '-')
    
    # 2. Замінюємо 3 і більше перенесень рядка на подвійне (абзац)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # 3. Замінюємо 2 і більше пробілів на один
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip()

# Для прикладу візьмемо одне з резюме
from connectors.resume_loader import load_resume_text
raw_text = load_resume_text("../data/Kostiantyn_Zivenko_Resume_EN.pdf")

cleaned_text = sanitize_text(raw_text)

print("Довжина ДО:", len(raw_text))
print("Довжина ПІСЛЯ:", len(cleaned_text))
print("\n--- Перші 500 символів очищеного тексту ---\n")
print(cleaned_text[:500])

# Архітектура агента: Порти та Адаптери (Task 2)

Перш ніж писати наступний код, зупинимось і подивимось на картину зверху.

## Що ми будуємо (Контекст)

Наш агент — це система, яка стоїть між **пошукачем** і **зовнішнім світом** (сайти вакансій, пошта, LLM).

```mermaid
graph TD
    Candidate["🧑 Пошукач<br/><i>IT-фахівець</i>"]
    Agent["🤖 Job Agent<br/><i>Аналізує, оцінює, діє</i>"]
    DOU["🌐 DOU / Djinni<br/><i>Джерело вакансій</i>"]
    LLM["🧠 Gemini API<br/><i>Екстракція, матчинг</i>"]
    Gmail["📧 Gmail<br/><i>Листи, відгуки</i>"]

    Candidate -- "резюме, побажання" --> Agent
    Agent -- "шортліст, чернетки" --> Candidate
    Agent -- "парсить вакансії" --> DOU
    Agent -- "екстракція, оцінка" --> LLM
    Agent -- "відгуки, листи" --> Gmail

    style Agent fill:#4a90d9,color:#fff,stroke:#2d6cb4
    style Candidate fill:#5cb85c,color:#fff,stroke:#4a9a4a
    style DOU fill:#f0ad4e,color:#fff,stroke:#d49a3e
    style LLM fill:#d9534f,color:#fff,stroke:#c44340
    style Gmail fill:#5bc0de,color:#fff,stroke:#46a5c4
```

## Як це влаштовано всередині (Функціональні блоки)

Всередині агент розділений на **Ядро** (доменна логіка) і **Конектори** (адаптери). Між ними стоять **Порти** — абстрактні інтерфейси.

```mermaid
graph LR
    subgraph External ["☁️ Зовнішній світ"]
        DOU["🌐 DOU"]
        Gemini["🧠 Gemini API"]
        GmailExt["📧 Gmail"]
    end

    subgraph Connectors ["🔌 connectors/ — Адаптери"]
        RL["resume_loader.py<br/><i>PDF → текст</i>"]
        VS["vacancy_source.py<br/><i>httpx + bs4</i>"]
        LLMC["llm.py<br/><i>instructor + genai</i>"]
        EM["email.py<br/><i>IMAP / SMTP</i>"]
    end

    subgraph Core ["🧊 core/ — Ядро"]
        Models["models.py<br/><i>Pydantic-моделі</i>"]
        Ports["ports.py<br/><i>Protocol інтерфейси</i>"]
        Logic["logic.py<br/><i>Фільтр без LLM</i>"]
    end

    NB["📓 Jupyter Notebook<br/><i>Диригент</i>"]

    NB --> Models
    NB --> Logic
    NB --> RL
    NB --> VS
    NB --> LLMC

    VS -.->|реалізує| Ports
    EM -.->|реалізує| Ports
    LLMC --> Models
    RL --> Models

    VS --> DOU
    LLMC --> Gemini
    EM --> GmailExt

    style Core fill:#e8f5e9,stroke:#4caf50
    style Connectors fill:#e3f2fd,stroke:#2196f3
    style External fill:#fce4ec,stroke:#e91e63
    style NB fill:#fff3e0,stroke:#ff9800
```

## Патерн «Порти та Адаптери» (Hexagonal Architecture)

Ідея проста:
- **Ядро (`core/`)** містить бізнес-логіку і моделі даних. Воно **нічого не знає** про інтернет, Gmail чи DOU. Ядро визначає **порти** — інтерфейси (`Protocol`), які описують *що* йому потрібно: «хтось, хто вміє `fetch()` вакансії» або «хтось, хто вміє `send()` повідомлення».
- **Конектори (`connectors/`)** — це **адаптери**, які реалізують ці порти. `DouSource` реалізує порт `VacancySource`. Якщо завтра ми захочемо парсити Djinni — напишемо новий адаптер `DjinniSource`, а ядро **не зміниться жодним рядком**.

Чому це важливо:
1. **Тестованість** — ядро можна тестувати без мережі, просто підставивши фейковий адаптер.
2. **Розширюваність** — нове джерело вакансій або канал зв'язку = новий файл, а не переписування всього.
3. **Чіткий поділ відповідальності** — конектор не приймає рішень, ядро не робить HTTP-запитів.

## Моделі ядра (`core/models.py`)

У файлі `core/models.py` ми визначили Pydantic-моделі — «словник», яким думає наш агент:

| Модель | Призначення |
|---|---|
| `CandidateProfile` | Хто пошукач: стек, досвід, освіта, мови |
| `SearchPreferences` | Чого хоче: зарплата, формат, стоп-слова |
| `Vacancy` | Вакансія: стек, рівень, мови, формат |
| `MatchResult` | Наскільки вакансія підходить (0–100 + причини) |

> Повнота даних — не модель, а функція `core/logic.missing_fields()` (рахуємо алгоритмічно, без LLM).

Допоміжні: `LanguageSkill` (мова + рівень CEFR), `ExperienceEntry` (компанія, роль, досягнення), `Education` (ступінь, спеціальність).

## Порти (`core/ports.py`)

| Порт | Контракт |
|---|---|
| `VacancySource` | `fetch(limit) → list[str]` |
| `MessageSender` | `send(to, subject, body) → None` |
| `MessageReceiver` | `fetch_unread(limit) → list[IncomingMessage]` |

> **Увага:** Весь код моделей та портів ми написали в окремих файлах `core/models.py` та `core/ports.py`. Нижче ми лише імпортуємо їх і перевіряємо, що все працює.

In [ ]:
from core.models import (
    CandidateProfile, SearchPreferences, Vacancy,
    ExperienceEntry, Education, LanguageSkill,
    MatchResult,
)
from core.ports import VacancySource, MessageSender, MessageReceiver

print("Моделі та порти імпортовано!")
print(f"Полів у CandidateProfile: {len(CandidateProfile.model_fields)}")
print(f"Полів у Vacancy: {len(Vacancy.model_fields)}")

# LLM-конектор: текст → структура (Task 3)

Ми маємо сирий текст резюме і Pydantic-моделі, які описують «що ми хочемо знати». Тепер потрібен **міст** між ними.

## Проблема

Класичний підхід (regex, xpath, шаблони) — **крихкий**: кожен новий формат резюме ламає парсер. А резюме бувають дуже різні — колонки, таблиці, вільна форма.

## Ідея: LLM як універсальний парсер

LLM вже «розуміє» текст. Нам потрібно лише сказати: «ось текст, ось схема — заповни». І воно працює.

## Як це зробити

### Наївний спосіб
```python
response = client.models.generate_content(model=..., contents=prompt)
profile = CandidateProfile.model_validate_json(response.text)
```
Проблеми: LLM може обгорнути JSON у markdown-блок, додати пояснення, повернути невалідні значення. Треба писати код очищення, ретраїв, валідації — **багато бойлерплейту**.

### Правильний спосіб: instructor
[instructor](https://github.com/jxnl/instructor) патчить SDK-клієнт і додає:
- `response_model` — Pydantic-модель як і схема для LLM, і валідатор
- **Автоматичні ретраї** — помилка валідації → LLM отримує фідбек і виправляє
- **Крос-провайдерність** — `from_genai()` / `from_openai()` / `from_provider("mistral/...")`

## Конектор `connectors/llm.py`

Ми вже написали його. Базова ідея — одна універсальна функція:

```python
# connectors/llm.py (ключова частина)
import instructor
from google import genai

_gemini = instructor.from_genai(genai.Client(api_key=os.environ["GOOGLE_API_KEY"]))

EXTRACT_MODEL = "gemini-3.1-flash-lite"     # екстракція — легка модель
JUDGE_MODEL   = "gemini-3.5-flash"           # судження — розумніша модель

def extract[T: BaseModel](text, schema: type[T], instruction, model=EXTRACT_MODEL) -> T:
    return _call_with_fallback(           # ← обгортка з ретраями та фолбеком (нижче)
        schema=schema, model=model,
        messages=[{"role": "user", "content": f"{instruction}\n\n---\n{text}"}],
    )
```

**Одна функція для всього:** резюме → `CandidateProfile`, вакансія → `Vacancy`, лист → `InterviewProposal`. Змінюється лише `schema` та `instruction`.

### Модель за задачею

Головний принцип вибору моделі — **відповідність моделі задачі**. Не треба брати найдорожчу модель «про всяк випадок»: парсинг — це механіка, а матчинг потребує reasoning.

| Тип задачі | Що робимо | Gemini (основна) | Mistral (фолбек) |
|---|---|---|---|
| **Екстракція** (парсинг) | дістати поля з тексту | `gemini-3.1-flash-lite` | `open-mistral-nemo` |
| **Судження** (reasoning) | достатність, матчинг, генерація | `gemini-3.5-flash` | `mistral-small-latest` |

Парсинг — механічна операція: проста/швидка модель дає той самий результат за менші гроші. Судження (наскільки вакансія підходить) потребує аналізу, тому модель важча. Цей принцип ми пронесемо через увесь агент.

## Надійність: коли модель недоступна (фолбек)

Ми працюємо на **free tier**, де моделі регулярно віддають `503 / 429 / UNAVAILABLE` (перевантаження, ліміти). Один виклик «в лоб» — лотерея. Тому `extract()` ходить не напряму, а через `_call_with_fallback`, який будує **ланцюг спроб** і перебирає його, доки хтось не відповість:

```
gemini-3.1-flash-lite → gemini-2.5-flash-lite → gemini-2.5-flash → open-mistral-nemo
     (основна)              (фолбек Gemini)        (фолбек Gemini)    (інший провайдер)
```

Логіка проста:
1. Пробуємо поточну модель кілька разів із **backoff** (пауза між спробами зростає).
2. Якщо вона стабільно недоступна — переходимо до наступної моделі в ланцюзі.
3. У **кінці кожного ланцюга стоїть Mistral** — окремий провайдер з власним ключем (`MISTRAL_API_KEY`) і **окремою квотою**. Коли весь безкоштовний Gemini лежить, він стає «останнім рятувальником».

> ⚙️ Важлива деталь: ретраїмо лише **серверні** помилки (`5xx`) і **rate limit** (`429`). Клієнтську помилку (`400`, `404` — неправильний запит) ретраї не виправлять, тому її кидаємо одразу.

Чому Mistral саме в кінці, а не одразу: спершу вичерпуємо безкоштовний Gemini й **бережемо квоту Mistral** на справді чорний день.

> **Чому це можливо в кілька рядків:** instructor — крос-провайдерний. Та сама `response_model` валідується однаково, хто б не відповів — Gemini чи Mistral. Для решти коду агента фолбек невидимий: `extract()` як повертав типізований об'єкт, так і повертає.
>
> ⚠️ Технічний нюанс залежностей: instructor очікує `mistralai` версії **1.x** (`from mistralai import Mistral`). SDK 2.x реорганізував цей імпорт і ламає `from_provider` — тому в проєкті пін `mistralai<2`.


In [ ]:
# ДЛЯ ЧОГО ЦЯ КЛІТИНКА: наочно показати ланцюг фолбеків — через які моделі
# (і яких провайдерів) піде запит по черзі, доки одна з них не відповість.
# Це той самий ланцюг, який _call_with_fallback перебирає всередині extract().
from connectors import llm

def show_chain(model: str) -> None:
    # відтворюємо ланцюг так само, як його будує _call_with_fallback:
    # основна модель (завжди Gemini) + її фолбеки
    chain = [(llm._gemini, model)] + llm._FALLBACKS.get(model, [])
    steps = [
        f"{name} [{'mistral' if client is llm._mistral else 'gemini'}]"
        for client, name in chain
    ]
    print(" → ".join(steps))

print("Ланцюг для ЕКСТРАКЦІЇ (парсинг):")
show_chain(llm.EXTRACT_MODEL)

print("\nЛанцюг для СУДЖЕННЯ (reasoning):")
show_chain(llm.JUDGE_MODEL)

print("\n💡 Останній у кожному ланцюзі — Mistral: інший провайдер, окрема квота,")
print("   «останній рятувальник», коли весь безкоштовний Gemini недоступний.")


In [ ]:
# Task 3: Екстракція профілю з першого резюме
from connectors.llm import extract
from core.models import CandidateProfile

profile = extract(
    resume_text,
    CandidateProfile,
    instruction=(
        "Витягни профіль кандидата з тексту резюме. "
        "Якщо поля нема — лиши None або порожній список."
    ),
)

print(f"👤 {profile.full_name}")
print(f"   Роль:     {profile.title}")
print(f"   Локація:  {profile.location}")
print(f"   Досвід:   {profile.years_experience} років ({profile.seniority})")
print(f"   Стек:     {profile.tech_stack}")
print(f"   Мови:     {[(l.language_code, l.level) for l in profile.languages]}")
print(f"   Освіта:   {[(e.degree, e.field) for e in profile.education]}")
print(f"\n📋 Резюме:   {(profile.summary or '')[:150]}...")


### Перевірка на іншому резюме

Головна перевага LLM-підходу — **один і той самий код працює на будь-якому форматі**. Спробуємо інше резюме:


In [ ]:
# Task 3 (продовження): Екстракція з іншого резюме — той самий код, інший файл
resume_text_2 = load_resume_text("../data/Rezjume_Kolpovs_ka_Marija.pdf")
cleaned_text_2 = sanitize_text(resume_text_2)

print(f"Текст другого резюме ({len(cleaned_text_2)} символів):")
print(cleaned_text_2[:300])
print("...\n")

profile_2 = extract(
    cleaned_text_2,
    CandidateProfile,
    instruction=(
        "Витягни профіль кандидата з тексту резюме. "
        "Якщо поля нема — лиши None або порожній список."
    ),
)

print(f"👤 {profile_2.full_name}")
print(f"   Роль:     {profile_2.title}")
print(f"   Досвід:   {profile_2.years_experience} років ({profile_2.seniority})")
print(f"   Стек:     {profile_2.tech_stack}")
print(f"\n💡 Один і той самий extract() — два різні резюме — два типізовані профілі.")


# Побажання пошукача: SearchPreferences (Task 4)

Резюме каже, **хто** ви. Але воно не каже, **чого ви хочете** від наступної роботи: яка зарплатна вилка влаштує, який формат (remote / hybrid / onsite), що для вас стоп-сигнал.

## Чому це окрема модель, а не екстракція

Цих даних **немає в жодному документі** — їх немає звідки «парсити». Їх задає сама людина. Тому тут **немає LLM**: ми просто інстанціюємо Pydantic-модель `SearchPreferences` значеннями в клітинці.

Це навмисно показує важливу річ: **частину контексту в агента вкладає людина**, а не модель. Згодом це стане містком до *approval gate* у частині 2 — агент діє в межах, які задав користувач.

## Що всередині

| Поле | Призначення |
|---|---|
| `desired_roles` | бажані ролі (за ними шукатимемо й оцінюватимемо) |
| `min_salary` | нижня межа вилки — нижче відсіємо ще без LLM |
| `work_formats` | прийнятні формати (`remote` / `hybrid` / `onsite`) |
| `locations` | бажані локації |
| `must_have` | обов'язкові умови |
| `deal_breakers` | стоп-слова: якщо є в описі — вакансія відпадає |

`work_formats` використовує `Enum` `WorkFormat` — значення обмежені трьома варіантами, тож помилитись у написанні («віддалена» замість `remote`) неможливо.


In [ ]:
# Task 4: побажання пошукача — те, чого НЕМАЄ в резюме.
# ЧОМУ без LLM: ці дані не існують у жодному документі — їх задає людина.
# Просто інстанціюємо модель значеннями. Це і є «контекст від людини».
from core.models import SearchPreferences, WorkFormat

prefs = SearchPreferences(
    desired_roles=["Python Backend Developer", "Backend Engineer", "AI/ML Engineer"],
    min_salary=4000,                                   # нижня межа вилки (USD/міс)
    # ЧОМУ Enum, а не рядок "remote": значення обмежене трьома варіантами —
    # неможливо випадково написати "віддалена" і тихо зламати фільтр нижче.
    work_formats=[WorkFormat.remote, WorkFormat.hybrid],
    locations=["Europe", "Remote"],
    must_have=["Python"],                              # без цього вакансію не розглядаємо
    deal_breakers=["unpaid", "internship", "стажування"],  # стоп-слова в описі вакансії
)

# Видимий результат: переконуємось, що модель зібралась і типи коректні
print("Бажані ролі:  ", prefs.desired_roles)
print("Від:          ", prefs.min_salary)
print("Формати:      ", [f.value for f in prefs.work_formats])
print("Локації:      ", prefs.locations)
print("Must-have:    ", prefs.must_have)
print("Deal-breakers:", prefs.deal_breakers)


# Джерело вакансій: DOU (Task 5)

Резюме і побажання — це «вхід від людини». Тепер агенту потрібен **зовнішній світ**: реальні вакансії. Беремо `jobs.dou.ua` — найбільший майданчик для України.

## Конектор «світ → LLM»

`DouSource` — це **адаптер**, що реалізує порт `VacancySource` (`fetch(limit) -> list[str]`). Усередині — звичайний HTTP у два кроки:

1. **Сторінка списку** → дістаємо посилання на вакансії (`httpx` + `BeautifulSoup`).
2. **Сторінка кожної вакансії** → дістаємо текст опису.

Кожен елемент результату — сирий текст оголошення **з URL на початку**. URL критичний: саме він стане `apply_url` (Task 6) і входом для «рук» агента в частині 2.

## Крихка точка: селектори

CSS-селектори — найненадійніша частина будь-якого скрапера: верстка сайту змінюється. Тому **ми звірили їх із реальним HTML наживо**, а не повірили документації:

| Що шукаємо | Селектор у `doc` | Реальність |
|---|---|---|
| лінки на вакансії | `div.vacancy div.title a.vt` | ❌ 0 збігів (застарів) → ✅ `a.vt` |
| текст вакансії | `.b-typo.vacancy-section` | ✅ працює |

> Це і є «Парсинг 1.0»: працює, доки не змінилась верстка. У Task 6 ми побачимо «Парсинг 2.0» — коли структуру з тексту дістає LLM, і їй байдуже до HTML-тегів.

## Більше вакансій: сторінки (пагінація)

За замовчуванням беремо лише першу сторінку списку (~20 карток). Наступні DOU дозавантажує через AJAX (POST на `xhr-load`), тож щоб узяти більше — задаємо діапазон сторінок:

```python
# перші три сторінки списку (≈60 карток), далі обмежуємо limit-ом
source = DouSource(start_page=1, stop_page=3)
raw_postings = source.fetch(limit=30)
```

`start_page=stop_page=1` (дефолт) = поведінка «як зараз». Сусідні AJAX-порції перекриваються — конектор дедуплікує лінки сам.

## Надійність на демо

DOU може заблокувати запит або «лягти» в ефірі. Тому `DouSource` кешує кожен успішний запит у `fixtures/dou_vacancies.json`. Прапорець `use_cache=True` читає знімок замість мережі — фолбек на випадок збою наживо.


In [ ]:
# Task 5: реальні оголошення з jobs.dou.ua
from connectors.vacancy_source import DouSource

# use_cache=False — живий запит (і одразу зберігає знімок у fixtures/ як фолбек).
# Якщо DOU недоступний на демо — досить замінити на DouSource(use_cache=True).
# ЧОМУ розширене вікно (2 сторінки, limit=15): конкретна вакансія повзе вниз
# списку від дня до дня. Беремо ширше, щоб вона гарантовано потрапила у вибірку.
source = DouSource(use_cache=False, start_page=1, stop_page=2)

raw_postings = source.fetch(limit=15)

print(f"Отримано оголошень: {len(raw_postings)}")
print("\n--- перше оголошення (URL + початок тексту) ---")
print(raw_postings[0][:600])

# Вакансії → list[Vacancy]: «Парсинг 2.0» (Task 6)

Ось і момент «ага». Структуру вакансії з тексту дістає **той самий `extract()`**, що й профіль із резюме — змінюється лише схема (`Vacancy`) та інструкція.

## Чому це сильно

«Парсинг 1.0» (Task 5) тримався на CSS-селекторах — і вже зламався (довелось міняти селектор на `a.vt`). «Парсинг 2.0» працює із **сенсом тексту**, а не з тегами: інструкції байдуже, як саме DOU зверстав сторінку.

| | Парсинг 1.0 (bs4) | Парсинг 2.0 (LLM) |
|---|---|---|
| на чому тримається | HTML-теги / класи | сенс тексту |
| ламається від | зміни верстки | майже нічого |
| код під новий сайт | новий парсер | та сама `extract`, інша інструкція |

## Два нюанси

- **`apply_url`** просимо LLM узяти з рядка `URL: ...` на початку тексту — це вхід для «рук» агента в частині 2.
- **`raw_text`** заповнюємо **кодом, а не LLM**: нам потрібен точний оригінал (для матчингу й листа), а не його переказ. Інструкція прямо каже моделі лишити це поле порожнім.


In [ ]:
# Task 6: кожне сире оголошення → типізована Vacancy тим самим extract()
from connectors.llm import extract
from core.models import Vacancy

VACANCY_INSTRUCTION = (
    "Витягни структуровані дані вакансії з тексту оголошення. "
    "Обов'язково знайди URL вакансії (apply_url) — він у рядку 'URL: ...' на початку. "
    "Визнач мову, якою написане оголошення (posting_language, ISO 639-1: 'uk', 'en'...). "
    "Поле raw_text залиш порожнім — його заповнить код. "
    "Якщо поля нема — None або порожній список."
)

vacancies: list[Vacancy] = []
for text in raw_postings:
    v = extract(text, Vacancy, instruction=VACANCY_INSTRUCTION)
    v.raw_text = text  # ЧОМУ кодом: потрібен точний оригінал, а не переказ від LLM
    vacancies.append(v)

# Видимий результат: таблиця типізованих вакансій (+ мова оголошення)
print(f"Структуровано вакансій: {len(vacancies)}\n")
for v in vacancies:
    fmt = v.work_format.value if v.work_format else "—"
    lang = v.posting_language or "?"
    print(f"{(v.company or '—')[:20]:20} | {(v.role or '—')[:30]:30} | {fmt:6} | {lang}")


# Алгоритмічний фільтр без LLM (Task 7)

## Головний принцип: спершу алгоритм, потім LLM

Усе, що можна вирішити **детерміновано**, вирішуємо звичайним кодом — і лише те, що справді потребує розуміння тексту, віддаємо LLM. Чому саме так:

- **Дешевше.** Перевірка стоп-слова чи числа — нуль токенів і нуль грошей. LLM-виклик коштує і час, і гроші. Відсіювати очевидно невідповідні вакансії моделлю — як наймати юриста, щоб викинути спам.
- **Надійніше.** LLM **стохастична за природою**: та сама вакансія двічі може дати різну відповідь, і навіть на очевидному є ненульова ймовірність помилки. А `if salary < min` помиляється рівно ніколи.
- **Швидше.** Код виконується миттєво; мережевий виклик LLM — це сотні мілісекунд, і він може впасти (503, ліміти).
- **Прозоріше.** Причина відсіву — це явний рядок коду, а не «чорна скринька». Її легко пояснити людині й налагодити.

Тому дорогу «думаючу» модель **бережемо на крок матчингу** (Task 9), де потрібен саме аналіз сенсу, а не порівняння фактів.

## Що перевіряємо детерміновано

| Перевірка | Логіка |
|---|---|
| стоп-слова | `deal_breaker` зустрічається в тексті оголошення |
| формат | формат вакансії відомий і не серед бажаних |
| вилка | `salary_max` відомий і нижчий за `min_salary` |
| обов'язкове | `must_have` skill узагалі не згадано в тексті |

> Тонкого «наскільки кандидат підходить» тут **немає навмисно** — це судження, а не факт. Грубо відсіювати за скіл-гепом кодом ризиковано: легко викинути придатну вакансію через синонім («JS» vs «JavaScript») чи формулювання. Саме тому це робота LLM-матчингу.
>
> Важливо: ми порівнюємо число лише коли воно **відоме**. Якщо вилки в оголошенні немає — це не відсів, а сигнал «бракує даних» для Task 8 (достатність).


In [ ]:
# Task 7: дешевий детермінований відсів — БЕЗ жодного виклику LLM
from core.logic import rejection_reason

survivors: list[Vacancy] = []
for v in vacancies:
    reason = rejection_reason(v, prefs, profile)
    if reason:
        print(f"✗ {(v.company or '—')[:20]:20} — {reason}")
    else:
        survivors.append(v)
        print(f"✓ {(v.company or '—')[:20]:20} — проходить далі")

print(f"\nПройшли фільтр: {len(survivors)} з {len(vacancies)}")

# Як ВИГЛЯДАЄ відсів (на вузькій релевантній вибірці часто проходять усі).
# Демонструємо механіку на штучному оголошенні зі стоп-словом:
demo = Vacancy(company="DemoCorp", raw_text="We offer an unpaid internship. Python, Django.")
print("\nДемо-відсів:", rejection_reason(demo, prefs, profile))


# Повнота даних: алгоритмічно (Task 8)

Вакансія пройшла дешевий фільтр (Task 7). Тепер питання: **чи достатньо в ній даних**, щоб брати її в шортліст на оцінку?

## Чому алгоритмічно, а не LLM

Тут діє той самий принцип, що в Task 7. Після Task 6 дані **вже структуровані** в поля `Vacancy`. Питання «чи є вилка / формат / стек» — це тепер просто `v.salary_max is None`. Це **факт, а не судження** → робимо кодом: дешево, миттєво, надійно. LLM мав би сенс для *сирого тексту* (зловити «competitive salary»), але ми свідомо спершу структуруємо — тож поля і є джерелом істини.

## Не ворота, а градація

Спокуса — зробити достатність жорсткими воротами (нема вилки → відсів). Натомість робимо м'якше й реалістичніше: вакансію з **невеликим gap** (1–2 незаповнені поля) **усе одно беремо** в шортліст. Чому: ідеально заповнених описів мало, а часткові часто варті уваги. Те, чого бракує (`missing`), ми **запам'ятовуємо** — і згодом просто **спитаємо в листі**.

| | Старий підхід | Новий |
|---|---|---|
| хто рахує | LLM | код (`core/logic`) |
| вакансія з gap | окрема черга «уточнення» | у шортліст, gap запам'ятовуємо |
| ефект gap | окремий «уточнюючий лист» | мотиваційний лист + **питання в кінці** (ч.2) |

## Функції (`core/logic.py`)

- `missing_fields(v, prefs)` → список незаповнених важливих полів (короткі мітки).
- `has_enough_info(v, prefs, max_gap=2)` → чи брати в шортліст (gap ≤ поріг).

> Так зникає окрема гілка «уточнюючий лист»: gap не виводить вакансію з гри, а лише додає питання в кінець мотиваційного листа. Менше коду, природніший потік.


In [ ]:
# Task 8: повнота даних — АЛГОРИТМІЧНО, без жодного виклику LLM
from core.logic import missing_fields, has_enough_info

# (вакансія, чого бракує) — кандидати в шортліст на матчинг (Task 9).
# gap зберігаємо разом із вакансією: він стане питаннями в листі (частина 2).
shortlist_candidates: list[tuple[Vacancy, list[str]]] = []
for v in survivors:
    gaps = missing_fields(v, prefs)
    if has_enough_info(v, prefs):          # невеликий gap — усе одно беремо
        shortlist_candidates.append((v, gaps))
        note = f"бракує: {gaps}" if gaps else "повні дані"
        print(f"✓ {(v.company or '—')[:20]:20} — у шортліст ({note})")
    else:
        print(f"✗ {(v.company or '—')[:20]:20} — замало даних: {gaps}")

print(f"\nУ шортлисті: {len(shortlist_candidates)} з {len(survivors)}")


# Матчинг: головне судження (Task 9)

Вакансії в шортлисті мають достатньо даних. Тепер — **головне питання всього пайплайна**: наскільки кожна підходить кандидату? Це вже **не факт, а судження** (зважити стек, рівень, контекст), тому тут — LLM.

## «Мислення» вмикається вибором моделі

Ми весь час тримали принцип «модель за задачею». Матчинг — саме той крок, де потрібна **reasoning-модель**: беремо `JUDGE_MODEL` (`gemini-3.5-flash`), а не легку extract-модель. Це і є «вмикання мислення» — не магічний прапорець, а свідомий вибір потужнішої моделі під складніше завдання.

> **Чому не `thinking_config`/`thinking_budget`?** Gemini дозволяє керувати «бюджетом мислення» через `GenerateContentConfig`. Але цей параметр **провайдер-специфічний** (genai-only) — він зламав би наш крос-провайдерний фолбек на Mistral. Тому «мислення» реалізуємо переносимо: вибором моделі, а не низькорівневим параметром одного провайдера.

## `MatchResult` — оцінка + прозорість

| Поле | Сенс |
|---|---|
| `score` | 0..100 — наскільки підходить |
| `reasons` | **чому** саме така оцінка — пояснення для людини |

`reasons` критичні: агент не просто видає число, а пояснює рішення — це робить шортліст придатним для людського перегляду (і для approval gate у частині 2).

## Результат кроку

Сортуємо за `score` — на виході **впорядкований шортліст із причинами**, де поряд зі score живе `gap` (чого бракує): саме він стане питаннями в листі (частина 2).


In [ ]:
# Task 9: матчинг — головне судження, через JUDGE_MODEL (reasoning-модель)
from connectors.llm import score_match

profile_summary = f"{profile.title}, {profile.seniority}, стек: {profile.tech_stack}"
prefs_summary = f"ролі: {prefs.desired_roles}, від {prefs.min_salary}"

# зберігаємо gap поряд зі score — він стане питаннями в листі (частина 2)
shortlist: list[tuple[Vacancy, MatchResult, list[str]]] = []
for v, gaps in shortlist_candidates:
    m = score_match(v.raw_text, profile_summary, prefs_summary)
    shortlist.append((v, m, gaps))

# впорядкований шортліст: найкращі збіги — згори
shortlist.sort(key=lambda t: t[1].score, reverse=True)

for v, m, gaps in shortlist:
    flag = f"  ⚠ уточнити: {gaps}" if gaps else ""
    print(f"{m.score:>3}  {(v.company or '—')[:20]:20} — {(v.role or '—')[:32]}{flag}")
    for r in m.reasons[:3]:
        print(f"       • {r}")


# Снапшот стану: міст у частину 2 (кінець частини 1)

Частина 1 завершена: у пам'яті ноутбука є `profile`, `prefs` і впорядкований `shortlist` (кожен елемент — `Vacancy` + `MatchResult` + `gap`). Але дві зустрічі — це різні дні й **різні ноутбуки**, а БД ми свідомо не заводили.

## Чому JSON, а не база

Стан тут маленький — це просто кілька об'єктів. Розгортати БД заради них — overkill. Тому серіалізуємо стан у звичайний JSON через `model_dump()` (Pydantic уміє це з коробки), а частина 2 його завантажить. Якщо файл не збережеться — не біда: частина 2 просто **пережене пайплайн ч.1 заново** (recap-блок).

## Кліфхенгер

Агент *бачить світ* і вже *знає, чого бракує* в кожній вакансії (`gap`) — але ще **не вміє нічого з цим зробити**: ні відгукнутися на DOU, ні спитати роботодавця про вилку, ні домовитись про зустріч.

У **частині 2** дамо йому «руки»: tool use, approval gate, мотиваційний лист із питаннями по `gap` і листування.


In [ ]:
# Кінець частини 1: серіалізуємо стан у JSON для частини 2 (не БД — лише model_dump)
import json
import pathlib

snapshot = {
    "profile": profile.model_dump(),
    "prefs": prefs.model_dump(),
    "shortlist": [
        {"vacancy": v.model_dump(), "match": m.model_dump(), "missing": gaps}
        for v, m, gaps in shortlist
    ],
}

# ноутбук лежить у notebooks/, тож шлях до data/ — на рівень вище
out = pathlib.Path("../data/state.json")
out.parent.mkdir(exist_ok=True)
# default=str — підстрахування на випадок несеріалізовних типів (напр. datetime)
out.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print(f"💾 Стан збережено: {out}")
print(f"   профіль:  {snapshot['profile']['full_name']}")
print(f"   у шортлисті: {len(snapshot['shortlist'])} вакансій")
